# Multichannel synthesis debug

Minimal notebook for a single two-channel Q/U synthesis. The target STs are computed from one compsep patch with Q/U cross-statistics, then one Q/U map is synthesized jointly.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

# Edit only this block for debugging.
MAP_PATH = Path("BICEP project/p71_m384_fft_f64_s2_r0_i0_p0_n0_bumpsteerable_c4d79664d0.npy")
BACKEND = "fft"              # "fft" or "kernel"
TARGET_PBC = False          # keep this False: the target comes from a patch
SYNTHESIS_PBC = True        # change this to False to synthesize without PBC
J = 5
L = 4
COMPUTE_PS = False
PS_METHOD = "legacy"      # "legacy" or "gaussian_rings"
MAX_ITER = 50
LR = 1.0
HISTORY_SIZE = 50
PRINT_ITER = 10
SEED = 26
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "STL_main").exists():
    REPO_ROOT = REPO_ROOT.parent if (REPO_ROOT.parent / "STL_main").exists() else REPO_ROOT
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import STL_main.torch_backend as bk
from STL_main.STL_2D_FFT_Torch import STL_2D_FFT_Torch
from STL_main.STL_2D_Kernel_Torch import STL_2D_Kernel_Torch
from STL_main.Synthesis import synthesize_from_stats

if BACKEND not in {"fft", "kernel"}:
    raise ValueError("BACKEND must be 'fft' or 'kernel'")
DataClass = STL_2D_FFT_Torch if BACKEND == "fft" else STL_2D_Kernel_Torch

device = torch.device(DEVICE)
bk._DEFAULT_DEVICE = device
bk._DEFAULT_DTYPE = torch.float32
bk._DEFAULT_COMPLEX_DTYPE = torch.complex64
torch.manual_seed(SEED)
np.random.seed(SEED)

print("backend:", BACKEND)
print("data class:", DataClass.__name__)
print("device:", device)
print("map:", MAP_PATH)


In [ ]:
x = np.load(MAP_PATH).astype(np.float32)
assert x.shape[0] == 2, f"expected (2, H, W), got {x.shape}"

q, u = x[0], x[1]
p = np.sqrt(q * q + u * u)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), constrained_layout=True)
for ax, im, title, cmap in zip(axes, [q, u, p], ["Q target", "U target", "P target"], ["coolwarm", "coolwarm", "magma"]):
    vmax = np.percentile(np.abs(im[np.isfinite(im)]), 99)
    vmin = -vmax if title[0] != "P" else 0.0
    h = ax.imshow(im, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(h, ax=ax, fraction=0.046, pad=0.04)
plt.show()


In [ ]:
# Target is one batch element with two channels: (Nb, Nc, H, W) = (1, 2, H, W).
data_t = torch.from_numpy(x[None]).to(device=device, dtype=torch.float32)
data = DataClass(data_t, pbc=TARGET_PBC)

# Q-Q, Q-U, U-U statistics.
cross_matrix = torch.tensor([[1, 1], [0, 1]], dtype=torch.bool, device=device)

st_kwargs = dict(
    J=J,
    L=L,
    compute_PS=COMPUTE_PS,
    norm="store_ref",
)
if BACKEND == "fft":
    st_kwargs["power_spectrum_method"] = PS_METHOD
elif PS_METHOD != "legacy":
    print("PS_METHOD is FFT-only; ignoring it for the kernel backend.")

st_op = data.get_ST_op(**st_kwargs)

with torch.no_grad():
    target_stats = st_op.apply(
        data,
        norm="store_ref",
        norm_batch_mean=True,
        compute_cross_matrix=cross_matrix,
        compute_PS=COMPUTE_PS,
    )

# synthesize_from_stats expects these fields even when no pre-standardization was used.
target_stats.mean_pre_std = torch.zeros((target_stats.Nb, target_stats.Nc), device=device, dtype=torch.float32)
target_stats.std_pre_std = torch.ones((target_stats.Nb, target_stats.Nc), device=device, dtype=torch.float32)

mu = target_stats.to_flatten(mean_along_batch=True, keepnans=False, flatten_complex=True)
print("target batch/channels:", target_stats.Nb, target_stats.Nc)
print("flattened target length:", mu.numel())


In [ ]:
synth = synthesize_from_stats(
    target_stats=target_stats,
    nbatch=1,
    pbc_running=SYNTHESIS_PBC,
    running_shape=x.shape[-2:],
    mean_field=True,
    max_iter=MAX_ITER,
    lr=LR,
    history_size=HISTORY_SIZE,
    print_iter=PRINT_ITER,
    seed=SEED,
    verbose=True,
)

synth_np = synth.detach().cpu().numpy() if hasattr(synth, "detach") else np.asarray(synth)
print("synth shape:", synth_np.shape)


In [ ]:
q_s, u_s = synth_np[0], synth_np[1]
p_s = np.sqrt(q_s * q_s + u_s * u_s)

fig, axes = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True)
for row, maps, label in [
    (0, [q, u, p], "target"),
    (1, [q_s, u_s, p_s], "synth"),
]:
    for ax, im, name, cmap in zip(axes[row], maps, ["Q", "U", "P"], ["coolwarm", "coolwarm", "magma"]):
        vmax = np.percentile(np.abs(im[np.isfinite(im)]), 99)
        if vmax == 0:
            vmax = 1.0
        vmin = -vmax if name != "P" else 0.0
        h = ax.imshow(im, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(f"{name} {label}")
        ax.set_xticks([])
        ax.set_yticks([])
        fig.colorbar(h, ax=ax, fraction=0.046, pad=0.04)
plt.show()
